# Ball detection timeline (ATD)

Map every video frame to whether the Automatic Tracking Data (ATD) **sees the ball** or not.
The result is a timeline of contiguous detection / gap intervals, drawn with contrasting colours
in matplotlib.

Ball presence is not a flag in the file — every frame has a row, and a missing ball is a `NaN`
coordinate. A single toggle (`GRADED_ONLY`) switches analysis between the **graded halves** and
the **full match** — see [DATA.md](../../DATA.md) for the half bounds.

---

In [ ]:
%matplotlib inline


In [ ]:

import os, sys
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from kloppy import metrica


import games

FIELD_SEP = ':'
MISSING = 'N'  # first character of "NaN"

paths = games.asset_paths(source='tactical')
n_frames = games.GAMES[games.GAME]['n_video_frames']
fps = games.GAMES[games.GAME]['fps']

print('game:     ', games.GAME)
print('frames:   ', n_frames)
print('fps:      ', fps)
print('atd_raw:  ', paths['atd_raw'])
print('exists:   ', os.path.isfile(paths['atd_raw']))

In [ ]:
def load_ball_present(raw_path, n_frames):
    """Boolean mask over video frames: True where the ATD reports a ball position.

    Indexing matches the submission / tactical video (0 .. n_frames-1). The ATD's own
    frame_id counts from 1, so video_frame = frame_id - 1.
    """
    present = np.zeros(n_frames, dtype=bool)
    with open(raw_path) as handle:
        for line in handle:
            parts = line.split(FIELD_SEP)
            if len(parts) != 3:
                continue
            frame_id = int(parts[0])
            present[frame_id - 1] = parts[2][0] != MISSING
    return present


ball_present_full = load_ball_present(paths['atd_raw'], n_frames)
n_present_full = int(ball_present_full.sum())
print('full video — ball present: %d / %d frames (%.1f%%)' % (
    n_present_full, n_frames, 100 * n_present_full / n_frames))

## Scope: graded halves vs full match

Set `GRADED_ONLY` below, then re-run from this cell downward.

- `True` — only the graded halves (possession-relevant). Pre-match, halftime, and post-match are
  dropped from intervals and greyed on the timeline.
- `False` — the entire video, including ungraded stretches.

Half bounds from `games.half_video_blocks()` / [DATA.md](../../DATA.md):

| | first frame | last frame |
|---|---|---|
| **1st half** | 11,185 | 90,651 |
| **2nd half** | 110,808 | 190,197 |

In [ ]:
# Flip this and re-run from here downward.
GRADED_ONLY = True

half_blocks = games.half_video_blocks(games.GAME)  # [(1, 11185, 90651), (2, 110808, 190197)]
SCOPE_LABEL = 'graded halves' if GRADED_ONLY else 'full match'

in_play = np.zeros(n_frames, dtype=bool)
for half, first, last in half_blocks:
    in_play[first:last + 1] = True
    print('half %d: frames %d–%d  (%d frames)' % (half, first, last, last - first + 1))

# Frames included in the analysis scope.
analysis_mask = in_play if GRADED_ONLY else np.ones(n_frames, dtype=bool)
# Blocks used to cut intervals (per half, or one block for the whole video).
analysis_blocks = half_blocks if GRADED_ONLY else [(None, 0, n_frames - 1)]

ball_present = ball_present_full & analysis_mask

n_scope = int(analysis_mask.sum())
n_present = int(ball_present.sum())
n_absent = n_scope - n_present
print()
print('scope:         %s  (GRADED_ONLY=%s)' % (SCOPE_LABEL, GRADED_ONLY))
print('kept frames:   %d / %d (%.1f%%)' % (n_scope, n_frames, 100 * n_scope / n_frames))
print('ball present:  %d / %d (%.1f%%)' % (n_present, n_scope, 100 * n_present / n_scope))
print('ball absent:   %d / %d (%.1f%%)' % (n_absent, n_scope, 100 * n_absent / n_scope))

## Contiguous intervals

Collapse the per-frame boolean into runs: `(start_frame, length, present)`. With `GRADED_ONLY`,
intervals are built **per half** so a gap never spans halftime; with the full match, one run
covers the whole video.

In [ ]:
def runs(mask):
    """Yield (start, length, value) for each contiguous run in a 1-d boolean array."""
    if len(mask) == 0:
        return
    changes = np.flatnonzero(mask[1:] != mask[:-1]) + 1
    bounds = np.concatenate(([0], changes, [len(mask)]))
    for i in range(len(bounds) - 1):
        start, end = int(bounds[i]), int(bounds[i + 1])
        yield start, end - start, bool(mask[start])


def intervals_in_blocks(ball_mask, blocks):
    """Runs of present/absent inside each analysis block (video-frame indexing)."""
    out = []
    for half, first, last in blocks:
        for s, L, p in runs(ball_mask[first:last + 1]):
            out.append((first + s, L, p, half))
    return out


intervals = intervals_in_blocks(ball_present_full, analysis_blocks)
present_runs = [(s, L) for s, L, p, _ in intervals if p]
absent_runs = [(s, L) for s, L, p, _ in intervals if not p]

print('%d present intervals, %d absent intervals (%d total)' % (
    len(present_runs), len(absent_runs), len(intervals)))
print('longest present gap-free stretch: %d frames (%.1f s)' % (
    max(L for _, L in present_runs), max(L for _, L in present_runs) / fps))
print('longest missing stretch:          %d frames (%.1f s)' % (
    max(L for _, L in absent_runs), max(L for _, L in absent_runs) / fps))

## Timeline

`broken_barh` paints each interval as a horizontal bar. Green = ball detected, red = not.
When `GRADED_ONLY` is on, ungraded stretches are shaded grey and carry no ball intervals.

In [ ]:
COLOR_PRESENT = '#1a9850'  # green — ball detected
COLOR_ABSENT = '#d73027'   # red — ball missing
COLOR_EXCLUDED = '#bdbdbd'  # grey — ungraded (filtered out)


def excluded_runs(n_frames, half_blocks):
    """Contiguous video-frame spans outside the graded halves."""
    covered = np.zeros(n_frames, dtype=bool)
    for _, first, last in half_blocks:
        covered[first:last + 1] = True
    return [(s, L) for s, L, inside in runs(covered) if not inside]


def plot_ball_timeline(present_runs, absent_runs, n_frames, half_blocks=None,
                       graded_only=None, ax=None, title=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(16, 2.4))
    else:
        fig = ax.figure

    if half_blocks is None:
        half_blocks = games.half_video_blocks(games.GAME)
    if graded_only is None:
        graded_only = GRADED_ONLY

    legend = [
        Patch(facecolor=COLOR_PRESENT, label='ball detected'),
        Patch(facecolor=COLOR_ABSENT, label='ball missing'),
    ]
    if graded_only:
        ax.broken_barh(excluded_runs(n_frames, half_blocks), (0, 1),
                       facecolors=COLOR_EXCLUDED, linewidth=0)
        legend.append(Patch(facecolor=COLOR_EXCLUDED, label='excluded (ungraded)'))

    ax.broken_barh(present_runs, (0, 1), facecolors=COLOR_PRESENT, linewidth=0)
    ax.broken_barh(absent_runs, (0, 1), facecolors=COLOR_ABSENT, linewidth=0)

    for half, first, last in half_blocks:
        ax.axvline(first, color='black', ls='--', lw=0.9, alpha=0.7)
        ax.axvline(last, color='black', ls='--', lw=0.9, alpha=0.7)
        ax.text((first + last) / 2, 1.08, 'half %d' % half, ha='center', va='bottom',
                fontsize=9, transform=ax.get_xaxis_transform())

    ax.set_xlim(0, n_frames)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xlabel('video frame')
    ax.set_title(title or ('Ball detection timeline (%s)' % SCOPE_LABEL))
    ax.legend(handles=legend, loc='upper right', framealpha=0.9)
    return fig, ax


fig, ax = plot_ball_timeline(present_runs, absent_runs, n_frames, half_blocks)
plt.tight_layout()
plt.show()

# Zoom into a window

For viewing smaller intervals, select start frame (from begining of ATD anotation) and duration window (in seconds -> translated to frames @25fps). This is to show the finer structure

In [ ]:
# First-half kickoff = start of graded half 1 (see games.py / DATA.md).
ZOOM_START = half_blocks[0][1]  # 11185 frame
ZOOM_SECONDS = 120
ZOOM_END = ZOOM_START + ZOOM_SECONDS * fps

zoom_present = [(s, L) for s, L in present_runs if s < ZOOM_END and s + L > ZOOM_START]
zoom_absent = [(s, L) for s, L in absent_runs if s < ZOOM_END and s + L > ZOOM_START]

fig, ax = plot_ball_timeline(
    zoom_present, zoom_absent, n_frames, half_blocks,
    title='Ball detection — %d s from 1st-half kickoff (frames %d–%d)' % (
        ZOOM_SECONDS, ZOOM_START, ZOOM_END))
ax.set_xlim(ZOOM_START, ZOOM_END)

# Twin axis in seconds from kickoff for readability.
ax2 = ax.twiny()
ax2.set_xlim(0, ZOOM_SECONDS)
ax2.set_xlabel('seconds from kickoff')

# plt.tight_layout()
plt.show()

In [ ]:
def frame_to_video_seconds(frame):
    seconds = frame / 25
    minutes = round(seconds/60)
    hour = round(minutes/60)
    return f"{hour}:{minutes%60}:{seconds%60}"


def timestamp_to_frame(h,m,s):
    return (s + m*60+ h*3600)*25


In [ ]:
frame_to_video_seconds(110808)
# timestamp_to_frame(1,15,49)

In [ ]:
dataset = metrica.load_tracking_epts(meta_data=paths['atd_meta'], raw_data=paths['atd_raw'], limit=110809)

In [ ]:
from starter.utils import plot_frame

plot_frame(dataset, 110808)   # 1st half kickoff
plt.show()

## Interval table (optional peek)

The same runs as a small DataFrame — useful if you want to filter long gaps or export them.
With `GRADED_ONLY`, each row also carries which half it belongs to (`half` is `None` in full-match
mode).

In [ ]:
import pandas as pd

interval_df = pd.DataFrame([
    {
        'start_frame': s,
        'end_frame': s + L - 1,
        'n_frames': L,
        'duration_s': L / fps,
        'ball_detected': p,
        'half': half
    }
    for s, L, p, half in intervals
])

print(interval_df['ball_detected'].value_counts().rename({True: 'detected', False: 'missing'}))
interval_df.sort_values('n_frames', ascending=False).head(10)



## Missing intervals by length

How long are the gaps in the current scope? Plot the **number of missing-ball intervals** against
their **length** (in frames). Most gaps are short; a log-scaled count axis keeps longer stretches
readable.

In [ ]:
missing_lengths = interval_df.loc[~interval_df['ball_detected'], 'n_frames'].to_numpy()

# Exact count per length (frames) — the discrete distribution the axes ask for.
length_counts = (
    pd.Series(missing_lengths)
    .value_counts()
    .sort_index()
    .rename_axis('length_frames')
    .reset_index(name='n_intervals')
)
length_counts['length_s'] = length_counts['length_frames'] / fps

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# --- Full distribution: every distinct length ---
ax = axes[0]
ax.bar(length_counts['length_frames'], length_counts['n_intervals'],
       width=1.0, align='center', color=COLOR_ABSENT, linewidth=0)
ax.set_xscale('log')
ax.set_xlabel('interval length (frames)')
ax.set_ylabel('number of missing intervals')
ax.set_title('All missing intervals')
ax.set_xlim(0, length_counts['length_frames'].max() * 1.02) #1.02 is to provide a little bit of space for the diagram

# Twin x in seconds for the same bars.
ax_sec = ax.twiny()
ax_sec.set_xscale('log')
ax_sec.set_xlim(ax.get_xlim()[0] / fps, ax.get_xlim()[1] / fps)
ax_sec.set_xlabel('interval length (seconds)')

# --- Short gaps only: exact bars up to 2 s (where most mass sits) ---
SHORT_MAX = 2 * fps  # 50 frames at 25 fps
short = length_counts[length_counts['length_frames'] <= SHORT_MAX]

ax = axes[1]
ax.bar(short['length_frames'], short['n_intervals'],
       width=1.0, align='center', color=COLOR_ABSENT, edgecolor='white', linewidth=0.3)
ax.set_xlabel('interval length (frames)')
ax.set_ylabel('number of missing intervals')
ax.set_title('Missing intervals ≤ %.0f s (%d frames)' % (SHORT_MAX / fps, SHORT_MAX))
ax.set_xlim(0.5, SHORT_MAX + 0.5)

ax_sec = ax.twiny()
ax_sec.set_xlim(ax.get_xlim()[0] / fps, ax.get_xlim()[1] / fps)
ax_sec.set_xlabel('interval length (seconds)')

fig.suptitle('Missing-ball intervals (%s): count vs length' % SCOPE_LABEL, y=1.02)
plt.tight_layout()
plt.show()

print('missing intervals: %d' % len(missing_lengths))
print('median length:     %d frames (%.2f s)' % (
    np.median(missing_lengths), np.median(missing_lengths) / fps))
print('mean length:       %.1f frames (%.2f s)' % (
    missing_lengths.mean(), missing_lengths.mean() / fps))
print('share ≤ 1 s:       %.1f%%' % (100 * (missing_lengths <= fps).mean()))
print('share ≤ 2 s:       %.1f%%' % (100 * (missing_lengths <= 2 * fps).mean()))
print('share > 5 s:       %.1f%%' % (100 * (missing_lengths > 10 * fps).mean()))
length_counts.sort_values('n_intervals', ascending=False).head(10)

In [ ]:
missing_lengths

So there are kind of 2 regimes for the intervals where the ball is missing: Small ones (<2s or <50frames) and large ones (>10 frames)